# Connect an MCP tool

Ask an agent about an order using a tool served over MCP. This notebook includes a tiny server, so you can try it without setting up an external service.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/02_mcp.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai,mcp] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a6/liteagents-0.3.0a6-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Create a demo MCP server

This writes one tool into a temporary Python file. LiteAgents will start and stop the server for each run.

In [ ]:
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="liteagents-"))

import sys

mcp_server = workspace / "orders.py"
mcp_server.write_text('''
try:
    from mcp.server.mcpserver import MCPServer
except ImportError:
    from mcp.server.fastmcp import FastMCP as MCPServer

server = MCPServer("Orders")

@server.tool()
def lookup_order(order_id: str) -> str:
    """Look up an order's payment status and total."""
    return f"Order {order_id}: paid, total USD 12"

server.run()
''')
print("Created the demo orders server.")

## 4. Connect the server

Add the server under `mcp_servers`. `orders` is your name for this connection; its tool becomes `orders_lookup_order`.

In [ ]:
from liteagents import ProfileOptions, run

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
profile.mcp_servers = {
    "orders": {"command": sys.executable, "args": [str(mcp_server)]},
}

## 5. Ask about an order

The answer should report **paid, USD 12**, using the MCP tool’s response.

In [ ]:
result = await run(
    "Use orders_lookup_order for A123 and tell me its payment status and total.",
    profile=profile,
)
print(result.text)

Try another order ID, or change the server’s return value in step 3.

For an existing remote MCP server, replace the connection with `{"url": "https://your-server.example/mcp"}`. [MCP configuration](https://github.com/BerriAI/liteagents/blob/main/docs/profiles.md) covers authentication and tool selection.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)